**Isolation trees**

In [ ]:
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Load training data
train_data = pd.read_csv('train.csv', index_col='timestamp')

# Drop any NaN values
train_data.dropna(inplace=True)

# Extract relevant features
features = train_data

# You can create additional features if needed, such as rolling statistics, etc.

# Standardize features
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Split data into train and validation sets
X_train, X_val = train_test_split(scaled_features, test_size=0.2, random_state=42)

# Train Isolation Forest model
clf = IsolationForest(contamination=0.1, random_state=42)
clf.fit(X_train)

# Predict anomalies on validation data
y_pred_val = clf.predict(X_val)

# Predict anomalies on test data
test_data = pd.read_csv('test.csv', index_col='timestamp')
test_data.dropna(inplace=True)
test_features = test_data
scaled_test_features = scaler.transform(test_features)
y_pred_test = clf.predict(scaled_test_features)

# Assuming anomalies are labeled as -1 by Isolation Forest, you can identify them like this:
anomalies_indices_test = test_data.index[y_pred_test == -1]

# Print indices of anomalies in the test data
print("Indices of anomalies in test data:", anomalies_indices_test)

Note: </br>Adjust parameters </br>
Feature extraction & selection</br>Add evaluation metric

1. **n_estimators**:  Increasing  can improve the robustness of the model but may also increase training time and memory consumption.

2. **max_samples**: A smaller value can lead to faster training times but may result in less accurate models, especially for datasets with a large number of samples.

3. **contamination**: set the threshold for classifying samples as anomalies. Adjusting this parameter is essential to balance the trade-off between false positives and false negatives.

4. **max_features**: help control the model's complexity and prevent overfitting, especially for datasets with a large number of features.

5. **bootstrap**: Bootstrapping can introduce randomness and diversity into the trees, which may improve the model's performance.

6. **random_state**: This parameter controls the random seed used for random number generation. Setting a fixed random state ensures reproducibility of results.


**CART(Classification & regression trees)**

In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load normal and anomalous training data
train_normal_data = pd.read_csv('train_normal.csv', index_col='timestamp')
train_anomalous_data = pd.read_csv('train_anam.csv', index_col='timestamp')

# Create labels: 0 for normal data, 1 for anomalous data
train_normal_data['label'] = 0
train_anomalous_data['label'] = 1

# Combine normal and anomalous data while maintaining the temporal order
train_data = pd.concat([train_normal_data, train_anomalous_data])

# Separate features and labels
X = train_data.drop(columns=['label'])
y = train_data['label']

# Split data into train and validation sets while maintaining the temporal order
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, shuffle=False)

# Train Decision Tree classifier
clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, y_train)

# Predict labels on validation data
y_pred_val = clf.predict(X_val)

# Calculate accuracy on validation data
accuracy = accuracy_score(y_val, y_pred_val)
print("Validation Accuracy:", accuracy)

# Load test data
test_data = pd.read_csv('test.csv', index_col='timestamp')

# Predict labels on test data
y_pred_test = clf.predict(test_data)

# Find consecutive timestamps where behavior matches anomalous behavior for more than 30 timestamps
anomaly_indices = []
consecutive_count = 0
for timestamp, label in zip(test_data.index, y_pred_test):
    if label == 1:
        consecutive_count += 1
    else:
        if consecutive_count >= 30:
            anomaly_indices.extend(test_data.index[test_data.index.get_loc(timestamp) - consecutive_count:test_data.index.get_loc(timestamp)])
        consecutive_count = 0

print("Indices of anomalies in test data:", anomaly_indices)


Note:</br> Doesnt capture temporal variations (create lag features (values from previous time steps) or rolling statistics (e.g., moving averages) to capture temporal dependencies) </br></br> Hyperparameters:</br>

1. **Minimum Samples Split:** Increasing this value can reduce overfitting.

2. **Minimum Samples Leaf:** helps control overfitting by ensuring that each leaf node contains a minimum number of samples.

3. **Maximum Features:** Limiting the number of features can help reduce computational complexity and overfitting.

4. **Criterion:** measure the quality of a split. For classification, common options are Gini impurity and entropy

5. **Splitter:** 'best' selects the best split based on a chosen criterion, while 'random' selects the best random split.

6. **Maximum Depth:** A deeper tree may capture more complex patterns but is prone to overfitting.

**Dynamic Time Warping (DTW):**

In [ ]:
from dtw import *
import numpy as np

# Load test data
test_data = np.loadtxt('test.csv', delimiter=',')

# Load normal and anomalous data
normal_data = np.loadtxt('normal_data.csv', delimiter=',')
anomalous_data = np.loadtxt('anomalous_data.csv', delimiter=',')

# Compute DTW distances for normal data
normal_distances = []
for cut in normal_data:
    distance, _ = fastdtw(test_data, cut, dist=euclidean)
    normal_distances.append(distance)

# Compute DTW distances for anomalous data
anomalous_distances = []
slice_length = 100  # Define the length of the slice, adjust as needed
for i in range(0, len(anomalous_data) - slice_length + 1):
    slice_anomalous = anomalous_data[i:i+slice_length]
    distance, _ = fastdtw(test_data, slice_anomalous, dist=euclidean)
    anomalous_distances.append(distance)

# Set thresholds based on statistical properties of distances or domain knowledge
normal_threshold = np.mean(normal_distances) + np.std(normal_distances)
anomalous_threshold = np.mean(anomalous_distances) + np.std(anomalous_distances)

# Identify anomalies
for distance in anomalous_distances:
    if distance > anomalous_threshold and distance > normal_threshold:
        print("Anomaly detected!")
        print("DTW distance:", distance)


Note: </br> Adjust window size</br> Use stat to capture mean/median/mode of normal_distances and anomalous_distances. Now if normal_distance> normal_threshold or anomalous_distance< anomalous_threshold detect anamoly(can change thresholds if needed)

**KDE(Kernal Density Estimation)**

In [ ]:
from sklearn.neighbors import KernelDensity
import numpy as np

# Load normal, anomalous, and test data
normal_data = np.loadtxt('normal_data.csv', delimiter=',')
anomalous_data = np.loadtxt('anomalous_data.csv', delimiter=',')
test_data = np.loadtxt('test_data.csv', delimiter=',')

# Fit KDE to normal data
kde_normal = KernelDensity(kernel='gaussian', bandwidth=0.1)
kde_normal.fit(normal_data)

# Fit KDE to anomalous data
kde_anomalous = KernelDensity(kernel='gaussian', bandwidth=0.1)
kde_anomalous.fit(anomalous_data)

# Evaluate log probability density for test data
log_prob_test_normal = kde_normal.score_samples(test_data)
log_prob_test_anomalous = kde_anomalous.score_samples(test_data)

# Set thresholds based on statistical properties of log probability densities
normal_threshold = np.mean(log_prob_test_normal) - np.std(log_prob_test_normal)
anomalous_threshold = np.mean(log_prob_test_anomalous) - np.std(log_prob_test_anomalous)

# Identify anomalies
anomaly_indices_normal = np.where(log_prob_test_normal < normal_threshold)[0]
anomaly_indices_anomalous = np.where(log_prob_test_anomalous > anomalous_threshold)[0]

# Print indices of anomalies
print("Indices of anomalies (normal data):", anomaly_indices_normal)
print("Indices of anomalies (anomalous data):", anomaly_indices_anomalous)


Note:</br> threshold</br>

GRID SEARCH

In [ ]:
# Define parameter grid
param_grid = {
    'kernel': ['gaussian', 'tophat', 'epanechnikov', 'exponential', 'linear'],
    'bandwidth': [0.01, 0.1, 1.0],  # Adjust as needed
    'algorithm': ['auto', 'kd_tree', 'ball_tree', 'brute'],
    'leaf_size': [10, 20, 30],  # Adjust as needed
    'metric': ['euclidean', 'manhattan', 'chebyshev']
}

# Create KernelDensity estimator
kde = KernelDensity()

# Perform grid search with leave-one-out cross-validation
grid_search = GridSearchCV(kde, param_grid, cv=LeaveOneOut(), n_jobs=-1)
grid_search.fit(normal_data)

# Get best hyperparameters
best_params = grid_search.best_params_
print("Best hyperparameters:", best_params)

# Fit KernelDensity estimator with best hyperparameters
best_kde = KernelDensity(**best_params)
best_kde.fit(normal_data)

# Evaluate log probability density for test data
log_prob_test = best_kde.score_samples(test_data)


**VARIMA**

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.varmax import VARMAX
from sklearn.metrics import mean_squared_error

# Load normal and anomalous data
normal_data = pd.read_csv('normal_data.csv', index_col='timestamp')
anomalous_data = pd.read_csv('anomalous_data.csv', index_col='timestamp')

# Train VARIMA model on normal data
model = VARMAX(normal_data, order=(1, 1))
results = model.fit(disp=False)

# Forecast next value in the time series
forecast = results.forecast()

# Calculate deviation between predicted and actual value
deviation = forecast - normal_data.iloc[-1]

# Set threshold based on expected deviation under normal conditions
threshold_normal = np.std(deviation)

# Detect anomalies in normal data
if np.abs(deviation) > threshold_normal:
    print("Anomaly detected in normal data")

# Evaluate model performance on test data
test_data = pd.read_csv('test_data.csv', index_col='timestamp')
forecast_test = results.forecast(steps=len(test_data))

# Calculate deviation between predicted and actual value for test data
deviation_test = forecast_test - test_data

# Set threshold for anomalous data based on expected deviation
threshold_anomalous = np.std(deviation_test)

# Detect anomalies in test data
if np.abs(deviation_test) < threshold_anomalous:
    print("Anomaly detected in test data")


GRID SEARCH

In [ ]:
from statsmodels.tsa.statespace.varmax import VARMAX
from sklearn.metrics import mean_squared_error

# Define parameter grid
orders = [(1, 0), (2, 0), (1, 1)]  # AR and MA orders
trends = ['c', 't', 'ct']  # Trend types
seasonal_periods = [12, 24, 48]  # Seasonal periods

best_score = float('inf')
best_params = None

# Iterate over parameter grid
for order in orders:
    for trend in trends:
        for seasonal_period in seasonal_periods:
            # Train VARMAX model
            model = VARMAX(train_data, order=order, trend=trend, seasonal_periods=seasonal_period)
            results = model.fit(disp=False)

            # Evaluate model performance
            forecast = results.forecast(steps=len(test_data))
            mse = mean_squared_error(test_data, forecast)

            # Update best parameters if needed
            if mse < best_score:
                best_score = mse
                best_params = {'order': order, 'trend': trend, 'seasonal_periods': seasonal_period}

print("Best parameters:", best_params)


**STL(season, trend, residue)**

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import STL

# Load data
data = pd.read_csv('time_series_data.csv', index_col='timestamp')

# Perform STL decomposition
stl = STL(data, seasonal=13)  # Specify seasonal period if known
result = stl.fit()

# Extract residue component
residue = result.resid

# Set threshold for anomaly detection
threshold = np.std(residue)

# Identify anomalies
anomalies = data[residue > threshold]

# Print anomalies
print("Anomalies:")
print(anomalies)

Note: </br>
Use only for rhythmic data like heart data

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import STL

# Load normal and test data
normal_data = pd.read_csv('normal_data.csv', index_col='timestamp')
test_data = pd.read_csv('test_data.csv', index_col='timestamp')

# Perform STL decomposition for normal data
stl_normal = STL(normal_data, seasonal=13)  # Assuming seasonal period is known
result_normal = stl_normal.fit()

# Extract seasonal and trend components for normal data
seasonal_normal = result_normal.seasonal
trend_normal = result_normal.trend

# Perform STL decomposition for test data
stl_test = STL(test_data, seasonal=13)  # Assuming seasonal period is known
result_test = stl_test.fit()

# Extract seasonal and trend components for test data
seasonal_test = result_test.seasonal
trend_test = result_test.trend

# Check if seasonal and trend components match
seasonal_match = np.allclose(seasonal_normal, seasonal_test)
trend_match = np.allclose(trend_normal, trend_test)

print("Seasonal components match:", seasonal_match)
print("Trend components match:", trend_match)


Note: Tolerance with train data

**Autoencoders**

In [ ]:
from keras.layers import Input, Dense
from keras.models import Model

# Define autoencoder architecture
input_dim =  # Specify input dimensionality
encoding_dim =  # Specify size of the encoded representation
input_layer = Input(shape=(input_dim,))
encoded = Dense(encoding_dim, activation='relu')(input_layer)
decoded = Dense(input_dim, activation='sigmoid')(encoded)
autoencoder = Model(input_layer, decoded)

# Compile autoencoder
autoencoder.compile(optimizer='adam', loss='mse')

# Train autoencoder on normal data
autoencoder.fit(normal_data, normal_data, epochs=100, batch_size=32, shuffle=True, validation_split=0.2)

# Reconstruct both normal and test data
reconstructed_normal_data = autoencoder.predict(normal_data)
reconstructed_test_data = autoencoder.predict(test_data)

# Calculate reconstruction error
reconstruction_error_normal = np.mean(np.square(normal_data - reconstructed_normal_data), axis=1)
reconstruction_error_test = np.mean(np.square(test_data - reconstructed_test_data), axis=1)

# Set threshold for anomaly detection
threshold = np.mean(reconstruction_error_normal) + 3 * np.std(reconstruction_error_normal)

# Identify anomalies
anomaly_indices = np.where(reconstruction_error_test > threshold)[0]
print("Indices of anomalies:", anomaly_indices)


1. **Encoding Dimensionality**: the size of the latent space representation. A smaller encoding dimensionality may lead to more compact representations but may lose important information, while a larger encoding dimensionality may capture more details but can increase computational complexity and risk overfitting.

2. **Number of Layers and Neurons**:Deeper architectures with more neurons per layer can capture more intricate features but may also be prone to overfitting, especially with limited training data.

3. **Activation Functions**:  Common choices include ReLU (Rectified Linear Unit) for hidden layers and sigmoid or tanh for the output layer

4. **Loss Function**: Mean Squared Error (MSE) is often used for reconstruction tasks, but other loss functions like binary cross-entropy or cosine

5. **Optimizer**: optimizer algorithm and its hyperparameters (e.g., learning rate, momentum). optimizers include Adam, RMSprop, and stochastic gradient descent (SGD)

6. **Regularization Techniques**: Regularization methods such as dropout, L1/L2 regularization, and batch normalization can help prevent overfitting and improve the generalization ability of the autoencoder.

7. **Batch Size and Epochs**: Batch size determines the number of samples used in each iteration of training, while the number of epochs specifies the total number of training iterations. Finding the right balance between batch size and the number of epochs is essential for efficient training and convergence.

8. **Initialization**: affect the convergence speed and performance of the autoencoder. Common initialization techniques include random initialization, Xavier initialization, and He initialization.

9. **Learning Rate Schedule**: avoid getting stuck in local minima. Techniques like learning rate decay, adaptive learning rates, and cyclical learning rates can be beneficial.

10. **Early Stopping**: Early stopping is a regularization technique that halts the training process when the performance on a validation set starts to degrade, thus preventing overfitting.


TUNING??

In [ ]:
import optuna
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from keras.models import Sequential
from keras.layers import Dense

# Define objective function
def objective(trial):
    # Define hyperparameters to optimize
    encoding_dim = trial.suggest_int('encoding_dim', 2, 64)
    num_layers = trial.suggest_int('num_layers', 1, 3)
    num_neurons = trial.suggest_int('num_neurons', 8, 256)
    activation = trial.suggest_categorical('activation', ['relu', 'tanh', 'sigmoid'])

    # Build autoencoder model
    autoencoder = Sequential()
    autoencoder.add(Dense(num_neurons, activation=activation, input_shape=(input_dim,)))
    for _ in range(num_layers - 1):
        autoencoder.add(Dense(num_neurons, activation=activation))
    autoencoder.add(Dense(encoding_dim, activation=activation))
    autoencoder.add(Dense(input_dim, activation=activation))

    # Compile model
    autoencoder.compile(optimizer='adam', loss='mse')

    # Train model
    autoencoder.fit(X_train, X_train, epochs=10, batch_size=32, validation_split=0.2)

    # Calculate validation loss
    val_loss = autoencoder.evaluate(X_val, X_val)

    return val_loss

# Split data into train and validation sets
X_train, X_val = train_test_split(data, test_size=0.2, random_state=42)

# Run hyperparameter optimization
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=100)

# Print best hyperparameters and loss
best_params = study.best_params
best_loss = study.best_value
print("Best hyperparameters:", best_params)
print("Best validation loss:", best_loss)


**Subsequence outliers**

Detecting subsequence outliers involves identifying unusual patterns or sequences within a time series dataset. These outliers can represent abnormal behavior or anomalies that deviate from the expected patterns. Here's a general approach to detecting subsequence outliers in your data:

Define a Subsequence: Decide on the length and characteristics of the subsequences you want to analyze. Subsequences can be fixed-length segments of the time series data or variable-length sequences.

Feature Extraction: Extract relevant features from each subsequence. These features should capture the characteristics of the data that are relevant for detecting outliers. Common features include mean, standard deviation, trend, seasonality, spectral features, etc.

Modeling: Choose a suitable model or algorithm for detecting outliers in the feature space. This could be a distance-based approach (e.g., clustering, nearest neighbors), density-based approach (e.g., kernel density estimation, Gaussian mixture models), or machine learning-based approach (e.g., isolation forests, autoencoders).

Thresholding: Set a threshold for identifying outliers based on statistical properties of the feature space or domain knowledge. Outliers are data points or subsequences that fall outside of this threshold.

Anomaly Detection: Apply the chosen model to the dataset and identify subsequences that are classified as outliers.

In [ ]:
from sklearn.ensemble import IsolationForest
import numpy as np

# Load data
data = np.loadtxt('time_series_data.csv')

# Define parameters
subsequence_length = 50  # Length of subsequences
contamination = 0.01  # Expected proportion of outliers

# Generate subsequences
subsequences = [data[i:i+subsequence_length] for i in range(len(data) - subsequence_length + 1)]

# Extract features from subsequences (e.g., mean and standard deviation)
features = np.array([[np.mean(subseq), np.std(subseq)] for subseq in subsequences])

# Fit isolation forest model
model = IsolationForest(contamination=contamination)
model.fit(features)

# Predict outliers
outlier_predictions = model.predict(features)

# Find indices of outliers
outlier_indices = np.where(outlier_predictions == -1)[0]

print("Indices of subsequence outliers:", outlier_indices)